# Giai đoạn 1 — Dữ liệu v2
Chọn sản phẩm chỉ từ train; bỏ hai tuần biên không đầy đủ; dùng chung split theo ngày. Notebook này ghi bộ mới vào `processed_data/v2`. Cài `requirements.txt` nếu môi trường chưa đủ thư viện.

In [ ]:
from pathlib import Path
import os
import sys

# Set this only if more than one source-code dataset is attached.
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_ROOT_OVERRIDE to the folder containing src. Candidates: {sorted(candidates)}')
    CODE_ROOT = next(iter(candidates))
if not (CODE_ROOT/'src/config.py').is_file():
    raise FileNotFoundError('The selected code folder is not source v2')
sys.path.insert(0, str(CODE_ROOT))
from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('Code:', CODE_ROOT, 'Output:', WORK_ROOT)


In [ ]:
RAW_DATA_OVERRIDE = None
RAW_DIR = resolve_raw_dir(CODE_ROOT, RAW_DATA_OVERRIDE)
PROCESSED_DIR = WORK_ROOT/'processed_data/v2'
from src.config import DataConfig
CONFIG = DataConfig()  # 80/12/12 complete weeks; train-only thresholds 20 weeks, 50 transactions; 8,000 items
print(CONFIG)


## 1. EDA
Đọc giao dịch theo từng khối để hạn chế RAM. Lượng bán quan sát là đại diện cho nhu cầu; không suy ra tồn kho từ số 0.

In [ ]:
from src.data.eda import run_eda
eda = run_eda(RAW_DIR, output_dir=WORK_ROOT/'eda/v2')


## 2. Tổng hợp và chọn mẫu
Lưu riêng `weekly_sales_all.csv` và `weekly_sales_selected.csv`. Giữ nguyên file toàn bộ sản phẩm để thử lại sampling.

In [ ]:
from src.pipeline.run_stage1_stage2 import run_stage1_stage2
summary = run_stage1_stage2(RAW_DIR, PROCESSED_DIR, CONFIG)
display(summary)


## 3. Xuất bộ input cho Kaggle
Upload toàn bộ archive này thành một dataset. Giữ nguyên nội dung các CSV để hash khớp. Không chỉ upload selected_articles.csv.

In [ ]:
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(str(WORK_ROOT/'hm-stage1-v2'), 'zip', root_dir=PROCESSED_DIR)
display(FileLink(archive))
